In [24]:
from ortools.sat.python import cp_model
import math

def solve_robot_path():
    model = cp_model.CpModel()

    grid_size = 5
    steps = 3

    x = [model.NewIntVar(0, grid_size-1, f'x{i}') for i in range(steps+1)]
    y = [model.NewIntVar(0, grid_size-1, f'y{i}') for i in range(steps+1)]

    # Start & End
    model.Add(x[0] == 1)
    model.Add(y[0] == 1)

    model.Add(x[steps] == 4)
    model.Add(y[steps] == 4)

    # Correct diagonal movement
    for i in range(steps):
        dx = model.NewIntVar(-1, 1, f'dx{i}')
        dy = model.NewIntVar(-1, 1, f'dy{i}')

        model.Add(dx == x[i+1] - x[i])
        model.Add(dy == y[i+1] - y[i])

        model.AddAllowedAssignments(
            [dx, dy],
            [(-1,-1), (-1,1), (1,-1), (1,1)]
        )

   
    obstacles = []
    for i in range(steps+1):
        for ox, oy in obstacles:
            model.AddForbiddenAssignments([x[i], y[i]], [(ox, oy)])

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        path = [(solver.Value(x[i]), solver.Value(y[i])) for i in range(steps+1)]
        print(" Path:", path)
        print("Cost:", steps * math.sqrt(2))
    else:
        print(" No solution found!")

solve_robot_path()

 Path: [(1, 1), (2, 2), (3, 3), (4, 4)]
Cost: 4.242640687119286


In [35]:
#task 2

# variables : grid
# constraints :
# 1) 1= land
# 2) check 4 neighbors if 0 , then include in perimter

from ortools.sat.python import cp_model

def island_perimeter(grid):
    rows, cols = len(grid), len(grid[0])
    perimeter = 0

    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                # Check all 4 sides
                if i == 0 or grid[i-1][j] == 0:
                    perimeter += 1
                if i == rows-1 or grid[i+1][j] == 0:
                    perimeter += 1
                if j == 0 or grid[i][j-1] == 0:
                    perimeter += 1
                if j == cols-1 or grid[i][j+1] == 0:
                    perimeter += 1

    return perimeter


grid = [
    [0,1,1,0,0],
    [1,1,1,0,0],
    [0,1,0,0,1],
    [0,0,0,1,1],
    [0,0,0,0,0]
]

print("Perimeter:", island_perimeter(grid))

Perimeter: 20


In [38]:
#task3
# variables  :  city visited
# domains : route
# contraints :
# 1)All cities visited once
# 2) return to start
# 3) minimize the cost

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def solve_tsp():
    distance_matrix = [
        [0,29,20,21,16,31,100,12,4,31],
        [29,0,15,29,28,40,72,21,29,41],
        [20,15,0,15,14,25,81,9,23,27],
        [21,29,15,0,4,12,92,12,25,13],
        [16,28,14,4,0,16,94,9,20,16],
        [31,40,25,12,16,0,95,24,36,3],
        [100,72,81,92,94,95,0,90,101,99],
        [12,21,9,12,9,24,90,0,15,25],
        [4,29,23,25,20,36,101,15,0,35],
        [31,41,27,13,16,3,99,25,35,0]
    ]

    manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

   
    def distance_callback(from_index, to_index):
        return distance_matrix[
            manager.IndexToNode(from_index)
        ][
            manager.IndexToNode(to_index)
        ]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

   
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        index = routing.Start(0)
        route = []

        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))

        route.append(0)  # return to start
        print(" Optimal Route:", route)

    else:
        print(" No solution found!")

solve_tsp()

 Optimal Route: [0, 8, 7, 2, 1, 6, 5, 9, 3, 4, 0]
